---
title: "Data Cleaning"
format:
    html: 
        code-fold: false
---

<!-- After digesting the instructions, you can delete this cell, these are assignment instructions and do not need to be included in your final submission.  -->

{{< include instructions.qmd >}} 

# Code 

Provide the source code used for this section of the project here.

If you're using a package for code organization, you can import it at this point. However, make sure that the **actual workflow steps**—including data processing, analysis, and other key tasks—are conducted and clearly demonstrated on this page. The goal is to show the technical flow of your project, highlighting how the code is executed to achieve your results.

If relevant, link to additional documentation or external references that explain any complex components. This section should give readers a clear view of how the project is implemented from a technical perspective.

Remember, this page is a technical narrative, NOT just a notebook with a collection of code cells, include in-line Prose, to describe what is going on.

In [32]:
import duckdb
import pandas as pd
import numpy as np

from pathlib import Path


In [33]:
RAW_PATH = Path("../../data/raw-data/ravelry_patterns.jsonl")
DB_PATH = Path("../../data/processed-data/ravelry.duckdb")

In [34]:
def load_raw_patterns(con):
    con.execute(f"""
                CREATE OR REPLACE TABLE raw_patterns AS
                SELECT * FROM read_json('{RAW_PATH}',
                                        format = 'newline_delimited',
                                        sample_size = -1);
    """)

con = duckdb.connect(str(DB_PATH))
load_raw_patterns(con)
print(con.execute("SELECT COUNT(*) AS n_patterns FROM raw_patterns").fetchone())

(25298,)


In [35]:
con.execute(""" 
    SELECT *
    FROM (DESCRIBE raw_patterns)
""").df()


,column_name,column_type,null,key,default,extra
0,comments_count,BIGINT,YES,None,None,None
1,created_at,VARCHAR,YES,None,None,None
2,currency,VARCHAR,YES,None,None,None
3,difficulty_average,DOUBLE,YES,None,None,None
4,difficulty_count,BIGINT,YES,None,None,None
5,downloadable,BOOLEAN,YES,None,None,None
6,favorites_count,BIGINT,YES,None,None,None
7,free,BOOLEAN,YES,None,None,None
8,gauge,DOUBLE,YES,None,None,None
9,gauge_divisor,BIGINT,YES,None,None,None


In [36]:
raw_df = con.execute("SELECT * FROM raw_patterns").df()
raw_df.shape

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(25298, 54)

In [37]:
raw_df.isna().sum()

comments_count                 0
created_at                     0
currency                    3354
difficulty_average             0
difficulty_count           18849
downloadable                   1
favorites_count                0
free                           1
gauge                       9731
gauge_divisor               3602
gauge_pattern                129
gauge_repeats              25172
generally_available         1091
has_uk_terminology         15279
has_us_terminology         10841
id                             0
name                           0
pdf_url                     5052
permalink                      0
price                      13057
projects_count                 0
published                   4526
queued_projects_count          0
rating_average                 0
rating_count               18880
row_gauge                  11940
updated_at                     0
url                          146
yardage                     9291
yardage_max                12025
yarn_list_

In [38]:
raw_df = raw_df.drop(columns=["gauge_repeats","personal_attributes","volumes_in_library","unlisted_product_ids"])

In [40]:
print((raw_df["rating_average"] == 0).sum(), raw_df["rating_count"].isna().sum())

18880 18880


In [41]:
no_ratings = raw_df["rating_count"].isna() | (raw_df["rating_count"] == 0)
raw_df.loc[no_ratings, "rating_average"] = np.nan
raw_df["rating_count"] = raw_df["rating_count"].fillna(0)

In [44]:
raw_df["craft_name"] = raw_df["craft"].str.get("name")

uk = raw_df["has_uk_terminology"].astype("string").fillna("missing")
us = raw_df["has_us_terminology"].astype("string").fillna("missing")

pd.crosstab([uk, us], raw_df["craft_name"])

craft_name                             Crochet  Knitting  Loom Knitting  \
has_uk_terminology has_us_terminology                                     
False              False                  1654      5642             39   
                   True                   1779         3              0   
                   missing                   1         0              0   
True               False                   153         0              0   
                   True                    440         0              0   
                   missing                 266         1              0   
missing            False                     5         0              0   
                   True                   4698         3              0   
                   missing                1297      9166             60   

craft_name                             Machine Knitting  
has_uk_terminology has_us_terminology                    
False              False                             41  
                   True                               0  
                   missing                            0  
True               False                              0  
                   True                               0  
                   missing                            0  
missing            False                              0  
                   True                               0  
                   missing                           50

In [45]:
raw_df = raw_df.drop(columns=["has_uk_terminology","has_us_terminology"])

In [53]:
pd.crosstab(raw_df["price"].isna(), raw_df["free"], dropna=False)

paid_no_price = (raw_df["free"] == False) & raw_df["price"].isna()
raw_df.loc[paid_no_price, "download_location"].str.get("type").value_counts(dropna=False)
raw_df.loc[paid_no_price, "downloadable"].value_counts(dropna=False)

downloadable
False    6833
True      282
Name: count, dtype: Int64

In [55]:

raw_df = raw_df[raw_df["free"].notna()]          # drop the 1 pattern with unknown free/paid

is_free = raw_df["free"].fillna(False)
is_download = raw_df["downloadable"].fillna(False)

raw_df["availability"] = np.select(
    [is_free, is_download],
    ["free", "paid_download"],
    default="print_only",
)
raw_df["availability"].value_counts()

availability
paid_download    11618
print_only        7050
free              6629
Name: count, dtype: int64

In [56]:
raw_df.isna().sum()

comments_count                 0
created_at                     0
currency                    3354
difficulty_average             0
difficulty_count           18848
downloadable                   0
favorites_count                0
free                           0
gauge                       9731
gauge_divisor               3601
gauge_pattern                129
generally_available         1090
id                             0
name                           0
pdf_url                     5052
permalink                      0
price                      13057
projects_count                 0
published                   4525
queued_projects_count          0
rating_average             18879
rating_count                   0
row_gauge                  11940
updated_at                     0
url                          146
yardage                     9291
yardage_max                12025
yarn_list_type              8632
sizes_available               33
product_id                 13246
currency_s